[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/colab-slm-playground/blob/main/gemma4_vllm_chatbot_colab.ipynb)

# Gemma 4 31B (NVFP4) via vLLM

Runs [nvidia/Gemma-4-31B-IT-NVFP4](https://huggingface.co/nvidia/Gemma-4-31B-IT-NVFP4) on a Colab GPU using vLLM.

**Requires a high-VRAM GPU runtime.** Go to Runtime → Change runtime type → A100 GPU.
T4 (16 GB) is too small. L4 (24 GB) may work but is tight.

## 1 — Check GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → A100 GPU, then re-run."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print()
if vram_gb < 20:
    print("WARNING: This model needs ~18 GB VRAM + overhead.")
    print("A T4 (16 GB) will likely OOM. Switch to A100 or L4.")
elif vram_gb < 30:
    print("L4 detected. Should work, but may be tight. A100 is safer.")
else:
    print("Plenty of VRAM. Good to go.")

## 2 — Install vLLM

Takes 2–4 minutes. vLLM bundles its own CUDA kernels.

In [ ]:
!pip install -q vllm
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q "nvidia-modelopt[all]"
!curl -sL https://raw.githubusercontent.com/jonasneves/colab-slm-playground/main/chat_ui.py -o /content/chat_ui.py

## 3 — HuggingFace login

Gemma 4 is a gated model. You need to:
1. Accept the license at [google/gemma-4-31b-it](https://huggingface.co/google/gemma-4-31b-it)
2. Provide your HF token below

Uncomment **one** of the options.

In [ ]:
# Option A — token stored in Colab secrets (recommended)
# from google.colab import userdata
# from huggingface_hub import login
# login(token=userdata.get("HF_TOKEN"))

# Option B — paste token directly (never commit this)
# from huggingface_hub import login
# login(token="hf_...")

print("Skipped HF login. Uncomment one option above if needed.")

## 4 — Start vLLM server

Downloads the model (~18 GB) and starts an OpenAI-compatible API server in the background.
First run takes several minutes for the download.

In [ ]:
import subprocess, time, requests

MODEL_ID = "nvidia/Gemma-4-31B-IT-NVFP4"
PORT = 8000

print(f"Starting vLLM server for {MODEL_ID}...")
print("This will download the model on first run (several minutes).")
print()

proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_ID,
        "--quantization", "modelopt",
        "--tensor-parallel-size", "1",
        "--max-model-len", "4096",
        "--port", str(PORT),
        "--trust-remote-code",
    ],
    stdout=open("/content/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait for the server to become healthy
base_url = f"http://localhost:{PORT}"
for i in range(300):  # up to 5 minutes
    try:
        r = requests.get(f"{base_url}/health", timeout=2)
        if r.status_code == 200:
            print(f"Server ready at {base_url}")
            break
    except requests.ConnectionError:
        pass
    if proc.poll() is not None:
        print("Server process exited. Full log:")
        !cat /content/vllm.log
        raise RuntimeError("vLLM failed to start.")
    time.sleep(1)
    if i % 30 == 29:
        print(f"  Still loading... ({i + 1}s)")
else:
    print("Timed out waiting for server. Check logs:")
    !cat /content/vllm.log
    raise RuntimeError("vLLM did not become healthy in 5 minutes.")

## 5 — Launch Chat UI

In [ ]:
import requests, json
from chat_ui import build_chat_html, register_callback
from IPython.display import display, HTML

def generate(messages: list) -> str:
    resp = requests.post(
        f"{base_url}/v1/chat/completions",
        json={
            "model": MODEL_ID,
            "messages": messages,
            "max_tokens": 512,
            "temperature": 0.7,
        },
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()

register_callback(generate)
display(HTML(build_chat_html("Gemma 4 31B", "NVFP4 &middot; vLLM &middot; GPU")))
print("Chat ready. First response may be slower due to KV cache warm-up.")

---
### Notes

**What is NVFP4?**
NVIDIA's FP4 quantization, applied via [NVIDIA Model Optimizer](https://github.com/NVIDIA/TensorRT-Model-Optimizer). It stores each weight in 4 bits using a floating-point representation (rather than integer quantization like GPTQ/AWQ). This preserves more dynamic range per bit. The tradeoff: it requires vLLM with the `modelopt` quantization backend and an NVIDIA GPU with FP8/FP4 support.

**Why vLLM instead of llama-cpp-python?**
NVFP4 weights are in safetensors format, not GGUF. vLLM is the inference engine NVIDIA targets for this quantization. It also provides PagedAttention for efficient KV cache management, continuous batching, and an OpenAI-compatible API out of the box.

**GPU requirements**
The quantized model is ~18 GB. A T4 (16 GB VRAM) cannot fit it. An L4 (24 GB) leaves ~6 GB for the KV cache, which works for short conversations. An A100 (40 GB) is comfortable. The `--max-model-len 4096` flag caps the context window to keep KV cache memory bounded.

**Why `--tensor-parallel-size 1`?**
The NVIDIA example uses TP=8 for production throughput across 8 GPUs. Colab provides a single GPU, so TP=1. The model still runs correctly; throughput is just lower.

**Multimodal capabilities**
Gemma 4 supports image and video input, but this notebook is text-only. Adding vision support would require passing image data through the API's multimodal message format.